In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import pandas as pd
import datetime as dt

from rockyelevate.wrapper import Session as Elevate
from rockyelevate.utils import response_to_dataframe as elv_res_2_df

from rockyclickup.wrapper import Session as Clickup
from rockyclickup.utils import response_to_dataframe as rcu_res_2_df
from rockyclickup.models import Client


from utils import Services

In [ ]:
str_date_today = dt.datetime.now().strftime("%m%d%y")

In [ ]:
services = Services()

In [ ]:
org_df = services.get_org_df()
org_df['id'] = org_df['id'].astype(str)

In [ ]:
client_df = services.get_client_df()

In [ ]:
merge_df = services.merge_df(org_df, client_df)
merge_df = merge_df.rename(
    columns={
        "organization_status_type": "organization_status",
        "status.status":            "client_status",
        "id_x":                     "organization_id",
        "id_y":                     "client_id"
    }
)

In [ ]:
clickup_status_map = {
    "active": "ACTIVE",
    "open enrollment": "OPEN_ENROLLMENT",
    "terminated": "TERMINATED",
    "onboarding": "ONBOARDING",
    "offboarding": "OFFBOARDING",
    "absorbed": "ABSORBED"
}


merge_df['client_status'] = merge_df['client_status'].map(clickup_status_map)

In [ ]:
merge_df['has_cobra'] = merge_df['client_cobra'].apply(lambda x: len(x) > 2)

In [ ]:
status_combos = {}

for index, row in merge_df.iterrows():
    status_tuple = (row["organization_status"], row['client_status'])

    if status_tuple not in status_combos:
        status_combos[status_tuple] = 0

    status_combos[status_tuple] += 1



In [ ]:
statuses_to_change = merge_df.copy()

statuses_to_change = statuses_to_change[
    statuses_to_change.apply(
        lambda row:
            (
                row['organization_status'] == "ACTIVE"
                and row['client_status'] == "TERMINATED"
            ) or (
                row['organization_status'] == "TERMINATED"
                and row['client_status'] in [
                    "OFFBOARDING",
                    "ACTIVE",
                    "OPEN_ENROLLMENT"
                ]
                and not row['has_cobra']
            ),
        axis=1
    )
]


In [ ]:
statuses_to_change[["organization_status", "client_status", "has_cobra"]]

In [ ]:
status_map = {
    ('ACTIVE', 'TERMINATED'): "ACTIVE",
    ('TERMINATED', 'OFFBOARDING'): "TERMINATED",
    ('TERMINATED', 'ACTIVE'): "TERMINATED",
    ('TERMINATED', 'OPEN_ENROLLMENT'): "TERMINATED",
}


#  ('ACTIVE', 'ACTIVE'): 865,
#  ('ACTIVE', 'ONBOARDING'): 20,
#  ('ACTIVE', 'OPEN_ENROLLMENT'): 351,
#  ('PENDING', 'ONBOARDING'): 10,
#  ('PENDING', 'ACTIVE'): 15,
#  ('PENDING', 'OFFBOARDING'): 1,
#  ('PENDING', 'TERMINATED'): 1,
#  ('ACTIVE', 'TERMINATED'): 3,
#  ('ACTIVE', 'OFFBOARDING'): 3,
#  ('TERMINATED', 'TERMINATED'): 121,
#  ('TERMINATED', 'OFFBOARDING'): 24,
#  ('TERMINATED', 'ACTIVE'): 3,
#  ('TERMINATED', 'OPEN_ENROLLMENT'): 5,
#  ('ACTIVE', 'ABSORBED'): 1

# check if client has a cobra plan

In [ ]:
merge_df[merge_df.apply(lambda row: row['organization_status'] == "PENDING" and row['client_status'] == "TERMINATED", axis=1)]

In [ ]:
statuses_to_change[["rmr_code", "organization_id", "client_id", "organization_status", "client_status", "has_cobra"]]

In [ ]:
[c for c in statuses_to_change if "name" in c]

In [ ]:
statuses_to_change[["rmr_code", "name_y", "client_id", "client_status", "organization_id", "organization_status"]]

In [ ]:
statuses_to_change[["rmr_code", "name_y", "client_id", "client_status", "organization_id", "organization_status"]].to_csv("statuses_to_change.csv", index=False)